# Scenarios

In [1]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Import custom modules
import plotting
import utils

In [3]:
base_data_folder = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
folder_name = "elec_s_128_ES_PT_no_bat_limit"
data_folder = os.path.join(base_data_folder, folder_name)

In [4]:
input_data = utils.load_csv_files_from_folder(data_folder)
batteries = input_data["batteries"]
branches = input_data["branches"]
generators = input_data["generators"]
capacity_factors = input_data["capacity_factors"]
generator_costs = input_data["generator_costs"]
hourly_demand = input_data["hourly_demand"]
nodes = input_data["nodes"]
nodes

,x,y,country
bus,,,
ES1 0,-1.336936,37.857622,ES
ES1 1,-3.338053,42.904161,ES
ES1 2,-0.740836,41.700214,ES
ES1 3,-5.340267,37.243441,ES
ES1 4,-7.552214,42.691137,ES
ES1 5,-3.579551,40.127828,ES
ES1 6,-0.771890,39.438307,ES
ES1 7,-5.578100,40.986702,ES
ES1 8,1.630357,41.642177,ES


In [5]:
base_demand = hourly_demand.sum().sum()
base_demand

290278276.1728846

In [6]:
def print_as_TWh(value):
    """Convert a value to TWh and print it with 2 decimal places."""
    print(f"{value / 1e6:.2f} TWh")

In [7]:
print_as_TWh(base_demand)

290.28 TWh


In [8]:
short_hand_to_full_name = {
    "NT": "National Trend",
    "GA": "Global Ambition",
    "DE": "Distributed Energy",
}
nt_2030 = 3264 * 1e6
nt_2040 = 3963 * 1e6
ga_2040 = 3617 * 1e6
ga_2050 = 3887 * 1e6
de_2040 = 4057 * 1e6
de_2050 = 4373 * 1e6
historical_2018 = 2745 * 1e6
estimate_2025 = historical_2018 + (nt_2030 - historical_2018) * (2025 - 2018) / (
    2030 - 2018
)
estimate_2025

3047750000.0

In [9]:
years = [2025, 2030, 2040, 2050]

In [10]:
# Assemble raw values per scenario
raw = {
    "NT": {2025: estimate_2025, 2030: nt_2030, 2040: nt_2040},
    "GA": {2025: estimate_2025, 2040: ga_2040, 2050: ga_2050},
    "DE": {2025: estimate_2025, 2040: de_2040, 2050: de_2050},
}

# Desired year order
years = [2025, 2030, 2040, 2050]

# Create DataFrame and reindex rows
df = pd.DataFrame(raw).reindex(years)

# Divide each column by its 2025 value to get scalars
scenario_multiplier = df.div(df.loc[2025])
scenario_multiplier.loc[2025, "GA"] = pd.NA
scenario_multiplier.loc[2025, "DE"] = pd.NA
scenario_multiplier

,NT,GA,DE
2025,1.000000,NaN,NaN
2030,1.070954,NaN,NaN
2040,1.300304,1.186777,1.331146
2050,NaN,1.275367,1.434829


In [11]:
total_demands = scenario_multiplier * estimate_2025 / 1e6

In [12]:
# Define path and filename
folder = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers"
filename = "base_scenarios.csv"
full_path = os.path.join(folder, filename)

# Ensure the directory exists
os.makedirs(folder, exist_ok=True)

# Save to CSV (including the year as the index)
scenario_multiplier.to_csv(full_path, index=True)

print(f"Saved scenario_multiplier to {full_path}")

Saved scenario_multiplier to C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\base_scenarios.csv


In [13]:
scenario_multiplier.index.values

print(set(scenario_multiplier.index.values) == set(years))

True


In [14]:
# Build the dictionary
scenarios = {
    year: [name for name in scenario_multiplier.loc[year].dropna().index]
    for year in scenario_multiplier.index
}

print(scenarios)

{2025: ['NT'], 2030: ['NT'], 2040: ['NT', 'GA', 'DE'], 2050: ['GA', 'DE']}


In [15]:
scenario_multiplier.index.values.tolist()

[2025, 2030, 2040, 2050]

### Making scenario trees

In [16]:
from itertools import product

# Create a list of all combinations of years and scenarios
combinations = list(product(scenario_multiplier.index.values, scenarios.values()))

In [19]:
options_per_year = [scenarios[year] for year in years]
scenario_trees = list(product(*options_per_year))

In [20]:
scenario_trees

[('NT', 'NT', 'NT', 'GA'),
 ('NT', 'NT', 'NT', 'DE'),
 ('NT', 'NT', 'GA', 'GA'),
 ('NT', 'NT', 'GA', 'DE'),
 ('NT', 'NT', 'DE', 'GA'),
 ('NT', 'NT', 'DE', 'DE')]

In [21]:
from itertools import product

# Your input dictionary
scenarios = {2025: ["NT"], 2030: ["NT"], 2040: ["NT", "GA", "DE"], 2050: ["GA", "DE"]}

# Get the years in sorted order for consistent output
years = sorted(scenarios)

# Get the list of scenario options for each year
options_per_year = [scenarios[year] for year in years]

# Compute the Cartesian product to get all scenario tree combinations
scenario_trees = list(product(*options_per_year))

# Optional: format as list of dicts for clarity
formatted_trees = [dict(zip(years, combo)) for combo in scenario_trees]

# Print results
for tree in formatted_trees:
    print(tree)

{2025: 'NT', 2030: 'NT', 2040: 'NT', 2050: 'GA'}
{2025: 'NT', 2030: 'NT', 2040: 'NT', 2050: 'DE'}
{2025: 'NT', 2030: 'NT', 2040: 'GA', 2050: 'GA'}
{2025: 'NT', 2030: 'NT', 2040: 'GA', 2050: 'DE'}
{2025: 'NT', 2030: 'NT', 2040: 'DE', 2050: 'GA'}
{2025: 'NT', 2030: 'NT', 2040: 'DE', 2050: 'DE'}


In [23]:
options_per_year

[['NT'], ['NT'], ['NT', 'GA', 'DE'], ['GA', 'DE']]

In [24]:
# Process each tree
scenario_dfs = []
for i, tree in enumerate(formatted_trees):
    # Create a mask: True for entries that match the selected scenario, False otherwise
    mask = pd.DataFrame(
        False, index=scenario_multiplier.index, columns=scenario_multiplier.columns
    )
    for year, scen in tree.items():
        mask.loc[year, scen] = True

    # Apply the mask
    filtered_df = scenario_multiplier.where(mask)

    # Drop columns that are entirely NaN
    filtered_df = filtered_df.dropna(axis=1, how="all")

    scenario_dfs.append(filtered_df)

    # Optional: label and display
    print(f"\nScenario Tree {i + 1}: {tree}")
    print(filtered_df)


Scenario Tree 1: {2025: 'NT', 2030: 'NT', 2040: 'NT', 2050: 'GA'}
            NT        GA
2025  1.000000       NaN
2030  1.070954       NaN
2040  1.300304       NaN
2050       NaN  1.275367

Scenario Tree 2: {2025: 'NT', 2030: 'NT', 2040: 'NT', 2050: 'DE'}
            NT        DE
2025  1.000000       NaN
2030  1.070954       NaN
2040  1.300304       NaN
2050       NaN  1.434829

Scenario Tree 3: {2025: 'NT', 2030: 'NT', 2040: 'GA', 2050: 'GA'}
            NT        GA
2025  1.000000       NaN
2030  1.070954       NaN
2040       NaN  1.186777
2050       NaN  1.275367

Scenario Tree 4: {2025: 'NT', 2030: 'NT', 2040: 'GA', 2050: 'DE'}
            NT        GA        DE
2025  1.000000       NaN       NaN
2030  1.070954       NaN       NaN
2040       NaN  1.186777       NaN
2050       NaN       NaN  1.434829

Scenario Tree 5: {2025: 'NT', 2030: 'NT', 2040: 'DE', 2050: 'GA'}
            NT        GA        DE
2025  1.000000       NaN       NaN
2030  1.070954       NaN       NaN
2040      

In [25]:
import os

# Folder path where files will be saved
folder = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers"

# Ensure the folder exists
os.makedirs(folder, exist_ok=True)

# Save each DataFrame to a CSV file
for i, df in enumerate(scenario_dfs, start=1):
    filename = f"EVPI_{i}.csv"
    full_path = os.path.join(folder, filename)
    df.to_csv(full_path)
    print(f"Saved: {full_path}")

Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_1.csv
Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_2.csv
Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_3.csv
Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_4.csv
Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_5.csv
Saved: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\EVPI_6.csv


In [27]:
# Compute the average multiplier per year, ignoring NaNs
average_multiplier = scenario_multiplier.mean(axis=1, skipna=True)

# Convert to a new DataFrame
average_df = pd.DataFrame({"mean": average_multiplier})

# Display the result
print(average_df)

          mean
2025  1.000000
2030  1.070954
2040  1.272742
2050  1.355098


In [28]:
# Define the folder path
folder = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers"

# Create the full path for the output file
mean_path = os.path.join(folder, "mean.csv")

# Save the DataFrame
average_df.to_csv(mean_path)

print(f"Saved mean scenario multiplier to: {mean_path}")

Saved mean scenario multiplier to: C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\mean.csv
